In [1]:
import os
import re
import json
import math
import requests
from pathlib import Path
from dataclasses import dataclass
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

# PDF ingestion
from pypdf import PdfReader

# Embeddings
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from gensim.models import Word2Vec

# (Optional) local generator
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


# Distributed computing system. 
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession 
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, 
    LongType, FloatType, ArrayType
)

In [2]:
@dataclass
class RAGConfig:
    pdf_path: str = "course_material"

    # Chunking (simple word-based chunking)
    chunk_size_words: int = 200
    chunk_overlap_words: int = 50

    # Retrieval
    top_k: int = 4
    similarity_metric: str = "cosine"   # {"cosine", "euclidean"}
    embedding_strategy: str = "sbert"   # {"tfidf", "sbert"}

    # Embedding models
    sbert_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"

    # Generator (fallback: small, CPU-friendly)
    gen_model_name: str = "llama3.2:3b"
    max_new_tokens: int = 1000

cfg = RAGConfig()

In [3]:
# Init Park
if SparkContext._active_spark_context:
    SparkContext._active_spark_context.stop()
    print("SparkContext existant arrêté.")
    
conf = SparkConf() \
    .setAppName("IngestData") \
    .setMaster("local[*]")

sc = SparkContext(conf=conf)
print("Spark version:", sc.version)

Spark version: 3.3.2


# Quelques fonctions
- sc.parallelize(c, numSlices=None)
    Elle prend deux paramètres :
    - 1. c — la collection (obligatoire)
    N'importe quelle collection Python itérable : list, range, tuple, string, dict, etc

    - 2. numSlices — le nombre de partitions (optionnel)
    Contrôle en combien de morceaux les données sont découpées. Si non spécifié, Spark décide automatiquement selon le nombre de cœurs disponibles.

    - Ce qu'elle retourne
    Un RDD — sur lequel vous pouvez ensuite chaîner des transformations (map, filter, flatMap...) ou des actions (collect, reduce, count...).

The pipeline must keep traceability metadata for every extracted unit, including at least the source PDF
name, page number, chunk identifier, and word offsets when available. This metadata is required later for
citation support and error analysis.

# Module 1 — Ingest: extract text from a PDF → page-level text (+ metadata for citations)

## Converted PDF to page-level raw text

In [4]:
def get_paths_pdf_files(folder_path: str) -> List[str]:
    """
    Browse a folder and get path from each PDF files.

    Parameters
    ----------
    folder_path : str
        Path to the folder containing the PDF files.

    Returns
    -------
    results : list of string
    """

    folder = Path(folder_path)

    # return paths all pdf files in folder path
    return list( folder.glob("*.pdf") )

# --- quick sanity check (should not error once implemented) ---
pdf_paths = get_paths_pdf_files(cfg.pdf_path)
rdd = sc.parallelize(pdf_paths)

In [5]:
print(pdf_paths)

[PosixPath('course_material/1-intro.pdf'), PosixPath('course_material/2-spark.pdf'), PosixPath('course_material/3-stream_proc.pdf'), PosixPath('course_material/4-stream_algos.pdf'), PosixPath('course_material/5-nosql.pdf'), PosixPath('course_material/6-parallel.pdf'), PosixPath('course_material/9780521192248_excerpt.pdf'), PosixPath('course_material/A Neural Network Playground.pdf'), PosixPath('course_material/a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf'), PosixPath('course_material/a_ScalableMachineLearningAlgorithmsforBigDataAnalytics-ChallengesandOpportunities.pdf'), PosixPath('course_material/a_the-big-book-of-machine-learning-use-cases.pdf'), PosixPath('course_material/a_TowardScalableSystemsForBigDataAnalytics.pdf'), PosixPath('course_material/Finding Similar Items.pdf'), PosixPath('course_material/Gama_ACMCS_AdaptationCD_accepted (1).pdf'), PosixPath('course_material/Gama_ACMCS_AdaptationCD_accepted.pdf'), PosixPath('course_material/intro.pdf'), PosixP

### Différence entre map et flatMap
La différence fondamentale
map → 1 élément entrant = 1 élément sortant
flatMap → 1 élément entrant = N éléments sortants (la liste résultante est "aplatie")

In [6]:
def extract_pages_from_pdf(pdf_path: str) -> List[Dict]:
    # Read pdf file
    reader = PdfReader(pdf_path)

    # For each page
    pages = list()
    for i in range( len(reader.pages) ):
        # Extract the text
        page = {
            "pdf_name" : Path(pdf_path).name,
            "page_num" : i + 1,
            "text" : reader.pages[i].extract_text() or ""
        }
        # Add the text to the list of pages
        pages.append(page)

    return pages

# flatMap : Regroupe tout les pages de chaque pdf dans une seule liste
pages_rdd = rdd.flatMap(extract_pages_from_pdf)

# 2. **Preprocess / clean** noisy PDF text -  cleaned as needed
Common PDF extraction artifacts such as hyphenation across line breaks, spurious line breaks, repeated whitespace, empty pages, headers/footers when they harm retrieval, and duplicated boilerplate. 
All preprocessing and cleaning procedures must be documented in the report.

In [7]:
def clean_text(page_from_pdf: dict) -> dict:
    """Basic cleanup for PDF-extracted text."""
    text = page_from_pdf["text"]
    # 1. Supprimer les césures en fin de ligne
    text = text.replace("-\n", "")
    
    # 2. Remplacer les retours à la ligne restants par des espaces
    text = text.replace("\n", " ")
    
    # 3. Réduire les espaces multiples à un seul espace
    text = re.sub(r"\s+", " ", text)
    
    # 4. Supprimer les espaces en début et fin
    page_from_pdf["text"] = text.strip()
    
    return page_from_pdf

# map : traitement chaque page
clean_rdd = pages_rdd.map(clean_text)

In [8]:
# filter : supprimer les pages vides
#final_rdd = clean_rdd.filter(lambda x: len(x["text"]) > 50)

In [9]:
print(clean_rdd.count(), "pages extraites")
print(clean_rdd.first())

1662 pages extraites
{'pdf_name': '1-intro.pdf', 'page_num': 1, 'text': 'INFO-H515: BIG DATA: DISTRIBUTED MANAGEMENT Lecture 1: Introduction, HDFS, Map/Reduce Dimitris Sacharidis'}


# 3. **Chunk** the text into fixed-size windows with overlap

In [10]:
def word_based_chunk_page(page_from_pdf: dict) -> list:
    """Split one page into overlapping word chunks."""
    
    chunk_size_words = cfg.chunk_size_words 
    overlap_words = cfg.chunk_overlap_words

    words = page_from_pdf["text"].split()
    page_num = page_from_pdf["page_num"]
    
    chunks = []
    step = chunk_size_words - overlap_words
    
    if step <= 0:
        raise ValueError("overlap_words must be smaller than chunk_size_words")
    
    chunk_index = 0
    for start in range(0, len(words), step):
        end = start + chunk_size_words
        chunk_words = words[start:end]
        if not chunk_words:
            break
        chunks.append({
            "chunk_id": f"p{page_num}_c{chunk_index}",
            "pdf_name": page_from_pdf["pdf_name"],
            "page_num":  page_num,
            "text": " ".join(chunk_words)
        })
        chunk_index += 1
        if end >= len(words):
            break
    return chunks
# Build chunks
clean_chunk_rdd = clean_rdd.flatMap(word_based_chunk_page)

In [11]:
print(clean_chunk_rdd.count(), " chunks")
print(clean_chunk_rdd.first())

2982  chunks
{'chunk_id': 'p1_c0', 'pdf_name': '1-intro.pdf', 'page_num': 1, 'text': 'INFO-H515: BIG DATA: DISTRIBUTED MANAGEMENT Lecture 1: Introduction, HDFS, Map/Reduce Dimitris Sacharidis'}


# 4. **Embed**: TF-IDF (baseline) and/or SBERT (dense)

## TF-IDF
TF-IDF calcule les fréquences de termes globales. Il faut collecter tous les chunks avant d'éffectuer se calcule.

In [12]:
def build_tfidf_index(chunks: List[Dict]) -> Tuple[TfidfVectorizer, np.ndarray]:
    """Fit TF-IDF on chunk texts and return (vectorizer, doc_matrix).

    Hints
    -----
    - Collect texts: `texts = [c["text"] for c in chunks]`
    - Use `TfidfVectorizer(...).fit_transform(texts)` to build a sparse matrix `X`
      with shape `[n_chunks, |V|]`.
    - Return `(vectorizer, X)`.
    """
    # YOUR CODE HERE
    texts = [c["text"] for c in chunks]

    vectorizer = TfidfVectorizer()
    X = vectorizer.fit_transform(texts)

    return (vectorizer, X)

# TF-IDF embedding
# Collecter tous les chunks
chunks = clean_chunk_rdd.collect()

# Fit TF-IDF
tf_idf_vectorizer, X_tf_idf = build_tfidf_index(chunks)

print(f"{len(chunks)} chunks")
print(f"Matrice TF-IDF : {X_tf_idf.shape}")  # (n_chunks, vocab_size)

2982 chunks
Matrice TF-IDF : (2982, 18371)


## Word2Vec

In [13]:
def build_word2vec_index(chunks: List[Dict], vector_size: int = 100, 
                          window: int = 5, min_count: int = 2, 
                          epochs: int = 10) -> Tuple[Word2Vec, np.ndarray]:
    """
    Train a Word2Vec model on chunk texts and generate chunk-level embeddings
    by averaging word vectors.
    """
    # Tokeniser chaque chunk en liste de mots
    tokenized = [c["text"].lower().split() for c in chunks]
    
    # Entraîner le modèle Word2Vec
    model = Word2Vec(
        sentences=tokenized,
        vector_size=vector_size,  # dimension des vecteurs
        window=window,            # contexte (mots voisins)
        min_count=min_count,      # ignorer les mots trop rares
        workers=4,                # parallélisme interne gensim
        epochs=epochs
    )
    
    # Générer l'embedding de chaque chunk ( moyenne des vecteurs de ses mots)
    def embed_chunk(words):
        # Garder uniquement les mots connus du modèle
        known = [w for w in words if w in model.wv]
        if not known:
            # vecteur nul pour les chunks vide ou des mots inconnu 
            return np.zeros(vector_size, dtype=np.float32)
        return np.mean([model.wv[w] for w in known], axis=0).astype(np.float32)
    
    embeddings = np.array([embed_chunk(words) for words in tokenized])
    
    return model, embeddings

# Entraîner Word2Vec
w2v_model, w2v_embeddings = build_word2vec_index(chunks)
print(f"Matrice Word2Vec : {w2v_embeddings.shape}")

Matrice Word2Vec : (2982, 100)


## SBERT

De même pour SBERT, on doit l'executer sur l'ensemble des chunks, et non chunk pas chunk.

In [14]:
def build_sbert_index(chunks: List[Dict], model_name: str) -> Tuple[SentenceTransformer, np.ndarray]:
    """Compute dense embeddings for all chunks using a SentenceTransformer.

    What SBERT gives you
    --------------------
    A SentenceTransformer is a Transformer encoder + pooling head that outputs one
    fixed-size embedding per input text. These embeddings are designed so that
    cosine similarity correlates with semantic similarity.

    Hints
    -----
    - `model = SentenceTransformer(model_name)`
    - `emb = model.encode(texts, show_progress_bar=True, normalize_embeddings=True)`
    - Cast to float32: `emb = np.asarray(emb, dtype=np.float32)`
    - Return `(model, emb)` where `emb` has shape `[n_chunks, dim]`.
    """
    # YOUR CODE HERE
    texts = [c["text"] for c in chunks]

    model = SentenceTransformer(model_name)
    emb = model.encode(
        texts,
        show_progress_bar=True,
        normalize_embeddings=True
    )

    emb = np.asarray(emb, dtype=np.float32)

    return model, emb

# SBERT embedding
model_name = "all-MiniLM-L6-v2"  # léger et efficace, 384 dimensions
sbert_model, sbert_emb = build_sbert_index(chunks, model_name)

print(f"Matrice SBERT : {sbert_emb.shape}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/94 [00:00<?, ?it/s]

Matrice SBERT : (2982, 384)


# 5. **Store** a lightweight local index

Pour sauvegarder en paquet, on aura besoin de SparkSession, car c'est lui qui gère les DataFrames / Paquet. 

In [15]:
# Besoin d'une SparkSession pour les DataFrames/Parquet
spark = SparkSession.builder \
    .config(conf=sc.getConf()) \
    .getOrCreate()

def save_to_hdfs_parquet(chunks, embeddings, embedding_strategy, hdfs_output_dir):
    """
    Sauvegarde chunks + embeddings en Parquet sur HDFS au format
    """
    
    schema = StructType([
        StructField("id",                 LongType(),              False),
        StructField("source_pdf",         StringType(),            True),
        StructField("page_num",           IntegerType(),           True),
        StructField("chunk_id",           StringType(),            True),
        StructField("start_word",         StringType(),           True),
        StructField("end_word",           StringType(),           True),
        StructField("chunk_text",         StringType(),            True),
        StructField("embedding",          ArrayType(FloatType()),  True),
        StructField("embedding_strategy", StringType(),            True),
    ])
    
    records = []
    for i, (chunk, emb) in enumerate(zip(chunks, embeddings)):
        words = chunk["text"].split()
        records.append((
            i, # id
            chunk["pdf_name"], # "source_pdf"
            chunk["page_num"], # "page_num"           
            chunk["chunk_id"], # "chunk_id"
            words[0], #"start_word"
            words[-1], #"end_word"
            chunk["text"], #"chunk_text"
            [float(v) for v in emb], #"embedding"
            embedding_strategy, #"embedding_strategy"
        ))

    # Redistribuer dans Spark et sauvegarder en Parquet sur HDFS
    df = spark.createDataFrame(records, schema=schema)
    df.write.mode("overwrite").parquet(f"hdfs://namenode:8020/{hdfs_output_dir}")
    print(f"Sauvegardé : hdfs://namenode:8020/{hdfs_output_dir}")
    print(f"Schema :") 
    df.printSchema()

In [16]:
# Sauvegarde tfidf prend un peu trop de temps
#save_to_hdfs_parquet(chunks, X_tf_idf.toarray(), "tfidf",  "project/embeddings/tfidf") 

In [17]:
# Sauvegarde word2vec
#save_to_hdfs_parquet(chunks, w2v_embeddings,   "word2vec","project/embeddings/word2vec")

In [18]:
# Sauvegarde sbert
save_to_hdfs_parquet(chunks, sbert_emb, "sbert",  "project/embeddings/sbert")

Sauvegardé : hdfs://namenode:8020/project/embeddings/sbert
Schema :
root
 |-- id: long (nullable = false)
 |-- source_pdf: string (nullable = true)
 |-- page_num: integer (nullable = true)
 |-- chunk_id: string (nullable = true)
 |-- start_word: string (nullable = true)
 |-- end_word: string (nullable = true)
 |-- chunk_text: string (nullable = true)
 |-- embedding: array (nullable = true)
 |    |-- element: float (containsNull = true)
 |-- embedding_strategy: string (nullable = true)



6. **Retrieve** Top-*k* chunks for a query using similarity search

In [19]:
def cosine_similarity(query_vec: np.ndarray, doc_matrix: np.ndarray) -> np.ndarray:
    """Compute cosine similarity between query_vec and each row of doc_matrix."""
    # YOUR CODE HERE
    # Produit scalaire entre la requête et chaque document
    dot_products = doc_matrix @ query_vec

    # Normes
    query_norm = np.linalg.norm(query_vec)
    doc_norms = np.linalg.norm(doc_matrix, axis=1)

    # Pour éviter une division par zéro
    eps = 1e-10
    similarities = dot_products / (doc_norms * query_norm + eps)

    return similarities


def euclidean_distance(query_vec: np.ndarray, doc_matrix: np.ndarray) -> np.ndarray:
    """Compute Euclidean distance between query_vec and each row of doc_matrix."""
    # YOUR CODE HERE
    # S'assurer que query_vec est 1D
    query_vec = query_vec.reshape(1, -1)

    # Différence vectorisée : (n_docs, dim)
    diff = doc_matrix - query_vec

    # Distance euclidienne par document
    distances = np.sqrt(np.sum(diff ** 2, axis=1))

    return distances


def retrieve_top_k(
    query: str,
    chunks: List[Dict],
    doc_matrix,
    k: int,
    embedding_strategy: str,
    similarity_metric: str,
    tfidf_vectorizer: Optional[TfidfVectorizer] = None,
    sbert_model: Optional[SentenceTransformer] = None,
    w2v_model: Optional[SentenceTransformer] = None,
) -> List[Dict]:
    """Return Top-k retrieved chunks for the query."""
    # -----------------------------
    # 1. Embed the query
    # -----------------------------
    if embedding_strategy == "tfidf":
        if tf_idf_vectorizer is None:
            raise ValueError("tfidf_vectorizer must be provided for TF-IDF retrieval")
        query_vec = tfidf_vectorizer.transform([query]).toarray()[0]

    elif embedding_strategy == "sbert":
        if sbert_model is None:
            raise ValueError("sbert_model must be provided for SBERT retrieval")
        query_vec = sbert_model.encode(
            query,
            normalize_embeddings=True
        )
        query_vec = np.asarray(query_vec, dtype=np.float32)
    
    elif embedding_strategy == "word2vec":
        if w2v_model is None:
            raise ValueError("w2v_model must be provided for Word2Vec retrieval")
        words = query.lower().split()
        known = [w for w in words if w in w2v_model.wv]
        if not known:
            raise ValueError("No known words in query")
        query_vec = np.mean(
            [w2v_model.wv[w] for w in known], axis=0
        ).astype(np.float32)
    
    else:
        raise ValueError(f"Unknown embedding_strategy: {embedding_strategy}")

    # -----------------------------
    # 2. Compute similarity / distance
    # -----------------------------
    if similarity_metric == "cosine":
        scores = cosine_similarity(query_vec, doc_matrix)
        # higher is better
        top_indices = np.argsort(scores)[::-1][:k]

    elif similarity_metric == "euclidean":
        scores = euclidean_distance(query_vec, doc_matrix)
        # lower is better
        top_indices = np.argsort(scores)[:k]

    else:
        raise ValueError(f"Unknown similarity_metric: {similarity_metric}")

    # -----------------------------
    # 3. Package results
    # -----------------------------
    results = []
    for idx in top_indices:
        results.append({
            "chunk": chunks[idx],
            "score": float(scores[idx])
        })

    return results

In [20]:
query = "What is Neural networks ?"

# Avec SBERT + cosine
results = retrieve_top_k(
    query=query,
    chunks=chunks,
    doc_matrix=sbert_emb,
    k=5,
    embedding_strategy="sbert",
    similarity_metric="cosine",
    sbert_model=sbert_model
)

for r in results:
    print(f"Score: {r['score']:.3f} | {r['chunk']['pdf_name']} p.{r['chunk']['page_num']}")
    print(f"  {r['chunk']['text'][:150]}...")
    print()

Score: 0.627 | a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf p.117
  activation function of the hidden layer neurons vector Z = weights between the hidden layer neurons and the output layer neurons func tion o= activati...

Score: 0.611 | A Neural Network Playground.pdf p.1
  Epoch 000,000 Learning rateActivation RegularizationRegularization rateProblem type Um, What Is a Neural Network? It’s a technique for building a comp...

Score: 0.587 | a_Big Data, Data Mining, and Machine Learning ( PDFDrive.com )-min.pdf p.117
  COMMON PREDICTIVE MODELING TECHNI Q UES ◂ 93 where W = weight vector to be computedW X = vector of known input variablesX y = known output A neural ne...

Score: 0.583 | A Neural Network Playground.pdf p.3
  Source on GitHub (https://github.com/tensorflow/playground) Credits This was created by Daniel Smilkov and Shan Carter. This is acontinuation of many ...

Score: 0.578 | Intro_DL_H515_2026.pdf p.5
  Deep learning = Neural networks 5 Google t

7. **Construct prompts** that ground the model and request citations

In [21]:
def build_exam_question_prompt(
    topic_query: str,
    retrieved: List[Dict],
    n_questions: int = 3,
) -> str:
    """Build a prompt that asks an LLM to generate exam questions grounded in sources."""
    # YOUR CODE HERE
    # Instructions principales
    prompt = f"""You are an expert teaching assistant.

TASK
Generate {n_questions} exam questions about:
"{topic_query}"

GROUNDING RULES (non-negotiable)
- Use ONLY the SOURCES provided below.
- If the sources are insufficient, say:
  "Insufficient information in the provided sources."
- Do NOT use outside knowledge.
- Every question and every answer MUST include citations like [1], [2], etc.
- Include:
  - at least one conceptual question
  - at least one question involving a formula or derivation if relevant

OUTPUT FORMAT
For i = 1..{n_questions}:
Q{{i}}. <question text> [citations]
A{{i}}. <short answer key> [citations]

SOURCES
"""

    # Ajout des chunks récupérés comme sources numérotées
    for i, r in enumerate(retrieved, start=1):
        c = r["chunk"]
        prompt += (
            f"\n[{i}] "
            f"(page {c['page_num']}, chunk {c['chunk_id']})\n"
            f"{c['text']}\n"
        )

    return prompt

In [22]:
topic = "Neural networks"
prompt = build_exam_question_prompt(topic, results, 5)
print(prompt)

You are an expert teaching assistant.

TASK
Generate 5 exam questions about:
"Neural networks"

GROUNDING RULES (non-negotiable)
- Use ONLY the SOURCES provided below.
- If the sources are insufficient, say:
  "Insufficient information in the provided sources."
- Do NOT use outside knowledge.
- Every question and every answer MUST include citations like [1], [2], etc.
- Include:
  - at least one conceptual question
  - at least one question involving a formula or derivation if relevant

OUTPUT FORMAT
For i = 1..5:
Q{i}. <question text> [citations]
A{i}. <short answer key> [citations]

SOURCES

[1] (page 117, chunk p117_c1)
activation function of the hidden layer neurons vector Z = weights between the hidden layer neurons and the output layer neurons func tion o= activation function of the output layer neurons Networks like these, generalized to include multi ple hidden la yers, are referred to as multilayer perceptrons (MLPs). However, the term “neural network” has been a pplied to a v

8. **Generate** questions with a free local model

In [23]:
import requests

GEN_BACKEND = "ollama"   # {"ollama"}
OLLAMA_MODEL = "llama3.2:3b"   # change if neede

def generate_text_ollama(prompt: str, model: str = OLLAMA_MODEL) -> str:
    """Generate text using a local Ollama model.

    Requires Ollama running locally: https://ollama.com
    """
    url = "http://host.docker.internal:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
    }
    r = requests.post(url, json=payload, timeout=120)
    r.raise_for_status()
    return r.json()["response"]

In [24]:
# Quick connectivity check (uncomment if using Ollama)
if GEN_BACKEND == "ollama":
    print(generate_text_ollama("Say hello in one sentence."))

Hello!


In [25]:
# Quick connectivity check (uncomment if using Ollama)
if GEN_BACKEND == "ollama":
    print(generate_text_ollama(prompt))

Here are 5 exam questions about "Neural Networks" based on the provided sources:

Q1. What is the primary difference between a neural network and other architectures, such as autoencoders or convolutional neural networks? [1]

A1. According to source [1], neural networks are typically used for two types of problems, which usually determines the number of neurons in the output layer, the activation function used by the output layer, and the error Figure 5.14 represents the output function yZW X oZ hW X(, , ) ( ( ) )=⋅ ⋅ where vector X = input variables vectorW = weights between the input layer and the hidden layer neurons func tion h = activation function of the hidden layer neurons vector Z = weights between the hidden layer neurons and the output layer neurons func tion o= activation function of the output layer neurons [1].

Q2. What is the role of regularization in a neural network? [4]

A2. According to source [4], regularization is used to prevent overfitting, which is often achie

# 9. **Evaluate** retrieval and generation with a tiny gold set + manual inspection

- Une évaluation superficielle ne sera pas considérer pour cette tâche.
- L'évaluation doit prendre en compte : retrieval, grounding, generation, and full-system behavior separately, following 
- Task : 
    - build a small evaluation set of representative course-related queries
        - Each query should contain gold evidence labels at the page or chunk level possible, together with expected topic, expected answer elements, or a grading rubic for the generates questions.  

## build a small evaluation set of representative course-related queries
- Pour construire cet ensemble, on survole les slides, et ont 

In [26]:
# Gold set for 1-intro.pdf
# INFO-H515 Lecture 1: Introduction, HDFS, MapReduce
GOLD_SET = [
    {"query": "What is Big Data and what are its main characteristics?", "relevant_pages": {15, 16, 17, 18, 19}, "pdf_name": "1-intro.pdf" },
    {"query": "How does parallelism help process large amounts of data?", "relevant_pages": {41, 42}, "pdf_name": "1-intro.pdf" },
    {"query": "What are the main challenges of distributed systems at scale?", "relevant_pages": {43, 44, 45, 46, 47}, "pdf_name": "1-intro.pdf" },
    {"query": "How does HDFS store and replicate data across nodes?", "relevant_pages": {51, 52, 53}, "pdf_name": "1-intro.pdf" },
    {"query": "What are the limitations and pitfalls of HDFS?", "relevant_pages": {53}, "pdf_name": "1-intro.pdf" },
    {"query": "How does the MapReduce programming model work?", "relevant_pages": {58, 59, 60, 61, 62, 63}, "pdf_name": "1-intro.pdf" },
    {"query": "How does Hadoop execute MapReduce jobs on a cluster?", "relevant_pages": {64, 65, 66, 67, 68}, "pdf_name": "1-intro.pdf" },
]

# Gold set for 2-spark.pdf
# INFO-H515 Lecture 2: RDD, Apache spark Eco System 
GOLD_SET += [
    {"query": "what is a Resilient Distributed Dataset", "relevant_pages": {9, 10, 11, 12, 13, 14}, "pdf_name": "2-spark.pdf"},
    {"query": "RDD transformations map and filter", "relevant_pages": {20, 21, 22, 23, 24, 25}, "pdf_name": "2-spark.pdf"},
    {"query": "lazy evaluation in Spark", "relevant_pages": {63, 64, 65}, "pdf_name": "2-spark.pdf"},
    {"query": "pair RDD reduceByKey aggregation", "relevant_pages": {54, 55, 56}, "pdf_name": "2-spark.pdf"},
    {"query": "caching and persistence storage levels", "relevant_pages": {70, 71, 72, 73}, "pdf_name": "2-spark.pdf"},
    {"query": "broadcast variables and accumulators", "relevant_pages": {45, 46}, "pdf_name": "2-spark.pdf"},
    {"query": "Spark master slave architecture driver executor", "relevant_pages": {75, 76, 77, 78}, "pdf_name": "2-spark.pdf"},
    {"query": "shuffled hash join partitioning", "relevant_pages": {58, 59, 60}, "pdf_name": "2-spark.pdf"},
    {"query": "Spark ecosystem MLlib Streaming GraphX SQL", "relevant_pages": {87, 88, 89, 90, 91}, "pdf_name": "2-spark.pdf"},
    {"query": "running Spark application on a cluster spark-submit", "relevant_pages": {82, 83, 84, 85}, "pdf_name": "2-spark.pdf"},
]

In [48]:
# Gold set for 1-intro.pdf
# INFO-H515 Lecture 1: Introduction, HDFS, MapReduce
GOLD_SET = [
    {
        "query": "What is Big Data and what are its main characteristics?",
        "relevant_pages": {15, 16, 17, 18, 19},
        "expected_topic": "Big Data definitions and the 3/4 Vs (volume, velocity, variety, veracity)",
        "expected_answer_elements": [
            "volume, velocity, variety",
            "datasets too large for traditional tools",
            "scalable processing"
        ]
    },
    {
        "query": "How does parallelism help process large amounts of data?",
        "relevant_pages": {41, 42},
        "expected_topic": "Maximal aggregate bandwidth and parallel data processing",
        "expected_answer_elements": [
            "aggregate bandwidth",
            "embarrassingly parallel",
            "multiple disks and CPUs in parallel"
        ]
    },
    {
        "query": "What are the main challenges of distributed systems at scale?",
        "relevant_pages": {43, 44, 45, 46, 47},
        "expected_topic": "Scalability and fault tolerance challenges",
        "expected_answer_elements": [
            "scalable software development",
            "fault tolerance",
            "redundancy and re-execution",
            "MTTF and disk failures"
        ]
    },
    {
        "query": "How does HDFS store and replicate data across nodes?",
        "relevant_pages": {51, 52, 53},
        "expected_topic": "HDFS architecture NameNode DataNode blocks replication",
        "expected_answer_elements": [
            "NameNode manages metadata",
            "DataNode stores blocks",
            "128MB block size",
            "rack-aware replication"
        ]
    },
    {
        "query": "What are the limitations and pitfalls of HDFS?",
        "relevant_pages": {53},
        "expected_topic": "HDFS limitations — large files, immutability, sequential access",
        "expected_answer_elements": [
            "optimized for large files",
            "files are immutable",
            "no random access",
            "append-only"
        ]
    },
    {
        "query": "How does the MapReduce programming model work?",
        "relevant_pages": {58, 59, 60, 61, 62, 63},
        "expected_topic": "MapReduce map and reduce functions key-value pairs",
        "expected_answer_elements": [
            "map function produces key-value pairs",
            "reduce aggregates values by key",
            "parallelism across tasks"
        ]
    },
    {
        "query": "How does Hadoop execute MapReduce jobs on a cluster?",
        "relevant_pages": {64, 65, 66, 67, 68},
        "expected_topic": "Hadoop V1 JobTracker TaskTracker execution and data locality",
        "expected_answer_elements": [
            "JobTracker schedules tasks",
            "TaskTracker executes tasks",
            "ship computation to data",
            "data locality"
        ]
    },
]

## Retrieval metrics : precision@k / recall@k (Section 2)

In [49]:
def precision_at_k(retrieved: List[Dict], relevant_pages: set, k: int) -> float:
    top = retrieved[:k]
    hits = sum(1 for r in top if r["chunk"]["page_num"] in relevant_pages)
    return hits / k

def recall_at_k(retrieved: List[Dict], relevant_pages: set, k: int) -> float:
    top = retrieved[:k]
    hit_pages = {r["chunk"]["page_num"] for r in top if r["chunk"]["page_num"] in relevant_pages}
    return len(hit_pages) / max(1, len(relevant_pages))

In [50]:
rows = []
for item in GOLD_SET:
    retrieved = retrieve_top_k(
        item["query"],
        chunks=chunks,
        doc_matrix=sbert_emb,
        k=cfg.top_k,
        embedding_strategy=cfg.embedding_strategy,
        similarity_metric=cfg.similarity_metric,
        tfidf_vectorizer=globals().get("tf_idf_vectorizer"),
        sbert_model=globals().get("sbert_model"),
    )
    p_at_k = precision_at_k(retrieved, item["relevant_pages"], k=cfg.top_k)
    r_at_k = recall_at_k(retrieved, item["relevant_pages"], k=cfg.top_k)
    rows.append({
        "query": item["query"],
        "P@k": p_at_k,
        "R@k": r_at_k,
        "hit_pages": sorted({r["chunk"]["page_num"] for r in retrieved}),
        "relevant_pages": item["relevant_pages"],
    })

df_eval = pd.DataFrame(rows)
df_eval

,query,P@k,R@k,hit_pages,relevant_pages
0,What is Big Data and what are its main charact...,0.75,0.600000,"[3, 15, 17, 18]","{16, 17, 18, 19, 15}"
1,How does parallelism help process large amount...,0.00,0.000000,"[2, 3, 6]","{41, 42}"
2,What are the main challenges of distributed sy...,0.25,0.200000,"[1, 5, 36, 44]","{43, 44, 45, 46, 47}"
3,How does HDFS store and replicate data across ...,0.50,0.666667,"[27, 51, 52, 74]","{51, 52, 53}"
4,What are the limitations and pitfalls of HDFS?,0.25,1.000000,"[49, 50, 53, 74]",{53}
5,How does the MapReduce programming model work?,0.00,0.000000,"[3, 19, 35, 36]","{58, 59, 60, 61, 62, 63}"
6,How does Hadoop execute MapReduce jobs on a cl...,1.00,0.800000,"[64, 65, 66, 67]","{64, 65, 66, 67, 68}"


## Generation and grounding metrics

In [51]:
#Définir le format de sortie attendu
# Format JSON attendu pour chaque génération
OUTPUT_SCHEMA = {
    "query": "str",
    "questions": [
        {
            "question": "str",
            "answer": "str", 
            "citations": ["pdf_name p.X chunk_id", ...]
        }
    ],
    "refusal": "str | null"  # si les sources sont insuffisantes
}

In [52]:
# Formater les sources numérotées
def sources_formatted(retrieved: List[Dict]) -> str:
    sources = ""
    for i, r in enumerate(retrieved):
        chunk = r["chunk"]
        sources += f"\n[{i+1}] (source: {chunk['pdf_name']}, page {chunk['page_num']}, chunk {chunk['chunk_id']})\n"
        sources += chunk["text"] + "\n"
    return sources

In [33]:
# Construire le prompt
def build_exam_question_prompt( topic_query: str, retrieved: List[Dict], n_questions: int = 3) -> str:
    """Build a prompt that asks an LLM to generate exam questions grounded in sources."""
    # Instructions principales
    prompt = f"""You are an expert teaching assistant.

TASK
Generate {n_questions} exam questions about:
"{topic_query}"

GROUNDING RULES (non-negotiable)
- Use ONLY the SOURCES provided below.
- If the sources are insufficient, say:
  "Insufficient information in the provided sources."
- Do NOT use outside knowledge.
- Every question and every answer MUST include citations like [1], [2], etc.
- Include:
  - at least one conceptual question
  - at least one question involving a formula or derivation if relevant

OUTPUT FORMAT (strict JSON):
{OUTPUT_SCHEMA}

SOURCES:
{sources_formatted(retrieved)}
"""
    return prompt

In [53]:
def build_exam_question_prompt(query: str, retrieved_chunks: list, n_questions: int = 3) -> str:
    
    sources = ""
    for i, r in enumerate(retrieved_chunks):
        chunk = r["chunk"]
        sources += (f"\n[{i+1}] source:{chunk['pdf_name']} "
                   f"page:{chunk['page_num']} "
                   f"chunk_id:{chunk['chunk_id']}\n")
        sources += chunk["text"] + "\n"

    prompt = f"""You are an exam question generator for a Big Data university course.

TASK: Generate {n_questions} exam questions about: "{query}"

STRICT RULES:
- Use ONLY information explicitly stated in the SOURCES below
- Do NOT invent facts, formulas, or definitions not present in the sources
- Each answer must be directly supported by at least one source
- If a source is insufficient, generate fewer questions or set refusal

SOURCES:
{sources}

For each citation, use the exact format: "pdf_name p.PAGE chunk_id:CHUNK_ID"
Example: "1-intro.pdf p.15 chunk_id:p15_c0"

Output ONLY valid JSON:
{{
  "query": "{query}",
  "questions": [
    {{
      "question": "...",
      "answer": "...",
      "citations": ["pdf_name p.PAGE chunk_id:CHUNK_ID"]
    }}
  ],
  "refusal": null
}}"""
    return prompt

In [54]:
# Appeler le LLM et parser la réponse
GEN_BACKEND = "ollama"   # {"ollama"}
OLLAMA_MODEL = "llama3.2:3b"   # change if neede

def generate_text_ollama(prompt: str, model: str = OLLAMA_MODEL) -> str:
    """Generate text using a local Ollama model.

    Requires Ollama running locally: https://ollama.com
    """
    url = "http://host.docker.internal:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.3,   # moins créatif = plus fidèle aux sources
            "num_predict": cfg.max_new_tokens,  # max tokens générés
        }
    }
    r = requests.post(url, json=payload, timeout=120)
    r.raise_for_status()
    return r.json()["response"]

In [55]:
def parse_llm_output(raw_output: str) -> dict:
    """
        Parse le JSON de la réponse LLM.
        Gère les cas où le LLM ajoute du texte autour du JSON.
        Retourne aussi un flag indiquant si le parsing a réussi.
    """
    # parsing direct
    try:
        return json.loads(raw_output.strip()), True
    except json.JSONDecodeError:
        pass
    
    # Extraire le JSON le premier { ... } dans le texte
    match = re.search(r"\{.*\}", raw_output, re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0)), True
        except json.JSONDecodeError:
            pass
    
    # retourner structure vide avec flag False
    return {
        "query": "",
        "questions": [],
        "refusal": f"PARSE_ERROR: {raw_output[:200]}"
    }, False

In [56]:
# Pipeline complet pour une query

In [57]:
import time
import psutil

def run_generation_pipeline(
    query: str,
    chunks: list,
    relevant_pages: list,
    doc_matrix,
    embedding_strategy: str,
    similarity_metric: str,
    k: int = 5,
    n_questions: int = 3,
    model: str = OLLAMA_MODEL,
    tfidf_vectorizer=globals().get("tf_idf_vectorizer") or None,
    sbert_model= globals().get("sbert_model") or None,
    w2v_model= globals().get("w2v_model") or None,
) -> dict:
    """
        Pipeline complet :    
    """
    process = psutil.Process()
    
    # 1. Retrieval
    t0 = time.time()
    retrieved = retrieve_top_k(
        query=query,
        chunks=chunks,
        doc_matrix=doc_matrix,
        k=k,
        embedding_strategy=embedding_strategy,
        similarity_metric=similarity_metric,
        tfidf_vectorizer=tfidf_vectorizer,
        sbert_model=sbert_model,
        w2v_model=w2v_model,
    )
    
    # Retrieval metrics
    p_at_k = precision_at_k(retrieved, relevant_pages, k=cfg.top_k)
    r_at_k = recall_at_k(retrieved, relevant_pages, k=cfg.top_k)
    
    retrieval_time = time.time() - t0
    
    # 2. Construction du prompt
    prompt = build_exam_question_prompt(query, retrieved, n_questions)
    tokens_in = len(prompt.split())   # approximation
    
    # 3. Génération LLM
    mem_before = process.memory_info().rss / 1e6  # MB
    t1 = time.time()
    raw_output = generate_text_ollama(prompt, model)
    generation_time = time.time() - t1
    mem_after = process.memory_info().rss / 1e6
    
    tokens_out = len(raw_output.split())  # approximation
    
    # 4. Parsing
    parsed_output, parse_success = parse_llm_output(raw_output)
    
    # 5. Assembler le résultat complet
    return {
        # Inputs
        "query":              query,
        "embedding_strategy": embedding_strategy,
        "similarity_metric":  similarity_metric,
        "k":                  k,
        "model":              model,
        
        # Retrieval
        "retrieved_chunks":   retrieved,
        "retrieval_time_s":   round(retrieval_time, 3),
        "p_at_k": p_at_k,
        "r_at_k": r_at_k,
        
        # LLM
        "prompt":             prompt,
        "raw_output":         raw_output,
        "tokens_in":          tokens_in,
        "tokens_out":         tokens_out,
        "generation_time_s":  round(generation_time, 3),
        "total_time_s":       round(retrieval_time + generation_time, 3),
        "memory_delta_mb":    round(mem_after - mem_before, 1),
        
        # Output parsé
        "parsed_output":      parsed_output,
        "parse_success":      parse_success,
        "n_questions":        len(parsed_output.get("questions", [])),
        "refusal":            parsed_output.get("refusal"),
    }

In [58]:
# Lancer sur toutes les queries du gold set
results = []

for eval_item in GOLD_SET:
    print(f"\nQuery: {eval_item['query']}")
    result = run_generation_pipeline(
        query=eval_item["query"],
        
        chunks=chunks,
        relevant_pages= eval_item["relevant_pages"],
        doc_matrix=sbert_emb,
        embedding_strategy=cfg.embedding_strategy,
        similarity_metric=cfg.similarity_metric,
        k=cfg.top_k,
        n_questions=3,
        sbert_model=sbert_model,
    )
    
    # Ajouter les gold labels pour l'évaluation
    result["gold_pages"] = eval_item["relevant_pages"]
    result["expected_topic"] = eval_item["expected_topic"]
    result["expected_answer_elements"] = eval_item["expected_answer_elements"]
    
    results.append(result)
    print(f"  ✅ {result['n_questions']} questions | "
          f"parse={'OK' if result['parse_success'] else 'FAIL'} | "
          f"time={result['total_time_s']}s")
    break
print(f"\nTotal : {len(results)} queries traitées")


Query: What is Big Data and what are its main characteristics?
  ✅ 3 questions | parse=OK | time=23.81s

Total : 1 queries traitées


In [59]:
# Inspecter le premier résultat
r = results[0]
print(f"Query     : {r['query']}")
print(f"Tokens in : {r['tokens_in']}")
print(f"Tokens out: {r['tokens_out']}")
print(f"Time      : {r['total_time_s']}s")
print(f"Parsed    : {r['parse_success']}")
print()
for q in r["parsed_output"].get("questions", []):
    print(f"Q: {q['question']}")
    print(f"A: {q['answer'][:150]}...")
    print(f"C: {q['citations']}")
    print()

Query     : What is Big Data and what are its main characteristics?
Tokens in : 380
Tokens out: 99
Time      : 23.81s
Parsed    : True

Q: What does the term 'Big Data' refer to in terms of its size?
A: extremely large datasets...
C: ['1-intro.pdf p.3 chunk_id:p3_c0']

Q: What are the three main characteristics of Big Data?
A: volume, variety, and velocity...
C: ['2-intro.pdf p.18 chunk_id:p18_c0']

Q: What challenges does Big Data pose in terms of data processing and analysis?
A: capturing data, data storage, data analysis, search, sharing, transfer, visualization, querying, updating and information privacy...
C: ['3-intro.pdf p.17 chunk_id:p17_c0']



In [60]:
r

{'query': 'What is Big Data and what are its main characteristics?',
 'embedding_strategy': 'sbert',
 'similarity_metric': 'cosine',
 'k': 4,
 'model': 'llama3.2:3b',
 'retrieved_chunks': [{'chunk': {'chunk_id': 'p3_c0',
    'pdf_name': '1-intro.pdf',
    'page_num': 3,
    'text': 'WHAT IS BIG DATA?'},
   'score': 0.9167785048484802},
  {'chunk': {'chunk_id': 'p18_c0',
    'pdf_name': '1-intro.pdf',
    'page_num': 18,
    'text': 'BIG DATA: SOME DEFINITIONS (4/4) Big data refers to extremely large datasets that cannot be easily managed, processed, or analyzed using traditional data processing methods. It is characterized by the three V’s: volume, variety, and velocity, which describe the sheer amount of data, the diversity of data types, and the speed at which it is generated. Big data technologies enable businesses and researchers to gain insights and make data-driven decisions from these vast, dynamic datasets. “ —ChatGPT (02/2025) 8'},
   'score': 0.7564034461975098},
  {'chunk': 

In [61]:
r.keys()

dict_keys(['query', 'embedding_strategy', 'similarity_metric', 'k', 'model', 'retrieved_chunks', 'retrieval_time_s', 'p_at_k', 'r_at_k', 'prompt', 'raw_output', 'tokens_in', 'tokens_out', 'generation_time_s', 'total_time_s', 'memory_delta_mb', 'parsed_output', 'parse_success', 'n_questions', 'refusal', 'gold_pages', 'expected_topic', 'expected_answer_elements'])

In [62]:
def check_citation_validity(citations: list, retrieved_chunks: list) -> dict:
    """
    Vérifie que les citations pointent vers des chunks réellement récupérés.
    """
    valid_ids = {r["chunk"]["chunk_id"] for r in retrieved_chunks}
    valid_pdfs = {r["chunk"]["pdf_name"] for r in retrieved_chunks}
    
    valid_count = 0
    for citation in citations:
        # Vérifier si le chunk_id ou le pdf_name apparaît dans la citation
        is_valid = any(cid in citation for cid in valid_ids) or \
                   any(pdf in citation for pdf in valid_pdfs)
        if is_valid:
            valid_count += 1
    
    return {
        "total_citations":  len(citations),
        "valid_citations":  valid_count,
        "citation_accuracy": valid_count / len(citations) if citations else 0.0
    }


def check_answer_grounding(answer: str, retrieved_chunks: list) -> float:
    """
    Mesure si la réponse est ancrée dans les chunks récupérés.
    Approche simple : overlap de mots-clés entre réponse et chunks.
    """
    answer_words = set(answer.lower().split())
    # Stopwords basiques
    stopwords = {"the", "a", "an", "is", "are", "was", "were", "in", 
                 "of", "to", "and", "or", "that", "this", "it", "its",
                 "as", "by", "for", "with", "on", "at", "from"}
    answer_words -= stopwords
    
    if not answer_words:
        return 0.0
    
    all_chunk_text = " ".join(r["chunk"]["text"] for r in retrieved_chunks).lower()
    chunk_words = set(all_chunk_text.split()) - stopwords
    
    overlap = answer_words & chunk_words
    return len(overlap) / len(answer_words)

"""
def check_answer_elements(answer: str, expected_elements: list) -> dict:
    #Vérifie si les éléments de réponse attendus sont présents.
    
    answer_lower = answer.lower()
    found = []
    missing = []
    
    for element in expected_elements:
        # Chercher les mots clés de l'élément dans la réponse
        keywords = element.lower().split()
        if any(kw in answer_lower for kw in keywords):
            found.append(element)
        else:
            missing.append(element)
    
    return {
        "found":   found,
        "missing": missing,
        "coverage": len(found) / len(expected_elements) if expected_elements else 0.0
    } 
"""

def check_answer_elements(answer: str, expected_elements: list) -> dict:
    """
    Version améliorée : matching par mots-clés individuels
    plutôt que par phrase entière.
    """
    answer_lower = answer.lower()
    found = []
    missing = []
    partial = []

    for element in expected_elements:
        keywords = [w for w in element.lower().split() 
                   if len(w) > 3]  # ignorer mots courts
        
        if not keywords:
            continue
            
        matches = sum(1 for kw in keywords if kw in answer_lower)
        ratio = matches / len(keywords)
        
        if ratio >= 0.6:        # au moins 60% des mots trouvés
            found.append(element)
        elif ratio >= 0.3:      # entre 30% et 60% → partiel
            partial.append(element)
        else:
            missing.append(element)

    total = len(expected_elements)
    # Score : found=1pt, partial=0.5pt
    score = (len(found) + 0.5 * len(partial)) / total if total > 0 else 0.0

    return {
        "found":    found,
        "partial":  partial,
        "missing":  missing,
        "coverage": round(score, 3)
    }


def evaluate_generation(result: dict) -> dict:
    """
    Évalue la qualité de génération pour un résultat du pipeline.
    Retourne les métriques pour chaque question générée.
    """
    questions     = result["parsed_output"].get("questions", [])
    retrieved     = result["retrieved_chunks"]
    expected_elem = result.get("expected_answer_elements", [])
    
    question_evals = []
    
    for q in questions:
        question_text = q.get("question", "")
        answer_text   = q.get("answer", "")
        citations     = q.get("citations", [])
        
        # Citation validity
        citation_check = check_citation_validity(citations, retrieved)
        
        # Answer grounding
        grounding_score = check_answer_grounding(answer_text, retrieved)
        
        # Expected answer elements coverage
        elements_check = check_answer_elements(answer_text, expected_elem)
        
        question_evals.append({
            "question":          question_text,
            "answer":            answer_text,
            "citations":         citations,
            # Métriques automatiques
            "citation_accuracy": citation_check["citation_accuracy"],
            "valid_citations":   citation_check["valid_citations"],
            "total_citations":   citation_check["total_citations"],
            "grounding_score":   round(grounding_score, 3),
            "elements_coverage": round(elements_check["coverage"], 3),
            "found_elements":    elements_check["found"],
            "missing_elements":  elements_check["missing"],
        })
    
    return {
        "query":            result["query"],
        "n_questions":      len(questions),
        "question_evals":   question_evals,
        "refusal":          result.get("refusal"),
        # Moyennes sur toutes les questions
        "avg_grounding":    round(
            sum(q["grounding_score"] for q in question_evals) / len(question_evals), 3
        ) if question_evals else 0.0,
        "avg_citation_acc": round(
            sum(q["citation_accuracy"] for q in question_evals) / len(question_evals), 3
        ) if question_evals else 0.0,
        "avg_elements_cov": round(
            sum(q["elements_coverage"] for q in question_evals) / len(question_evals), 3
        ) if question_evals else 0.0,
    }


# Évaluer tous les résultats
generation_evals = [evaluate_generation(r) for r in results]

# Afficher le résumé
print(f"{'Query':<50} | {'Grounding':>9} | {'Citation':>8} | {'Elements':>8}")
print("-" * 85)
for ev in generation_evals:
    print(f"{ev['query'][:50]:<50} | "
          f"{ev['avg_grounding']:>9.3f} | "
          f"{ev['avg_citation_acc']:>8.3f} | "
          f"{ev['avg_elements_cov']:>8.3f}")

print("-" * 85)
print(f"{'AVERAGE':<50} | "
      f"{sum(e['avg_grounding']    for e in generation_evals)/len(generation_evals):>9.3f} | "
      f"{sum(e['avg_citation_acc'] for e in generation_evals)/len(generation_evals):>8.3f} | "
      f"{sum(e['avg_elements_cov'] for e in generation_evals)/len(generation_evals):>8.3f}")

Query                                              | Grounding | Citation | Elements
-------------------------------------------------------------------------------------
What is Big Data and what are its main characteris |     0.863 |    1.000 |    0.167
-------------------------------------------------------------------------------------
AVERAGE                                            |     0.863 |    1.000 |    0.167


In [63]:
# Voir exactement ce qui manque
ev = generation_evals[0]
print(f"Query: {ev['query']}\n")

for i, q_eval in enumerate(ev["question_evals"]):
    print(f"Question {i+1}: {q_eval['question'][:80]}")
    print(f"  Grounding    : {q_eval['grounding_score']}")
    print(f"  Citations OK : {q_eval['valid_citations']}/{q_eval['total_citations']}")
    print(f"  Found        : {q_eval['found_elements']}")
    print(f"  Missing      : {q_eval['missing_elements']}")
    print()

# Vérifier les expected_answer_elements du gold set
print("Expected elements:")
for e in results[0]['expected_answer_elements']:
    print(f"  - {e}")

Query: What is Big Data and what are its main characteristics?

Question 1: What does the term 'Big Data' refer to in terms of its size?
  Grounding    : 1.0
  Citations OK : 1/1
  Found        : []
  Missing      : ['volume, velocity, variety', 'scalable processing']

Question 2: What are the three main characteristics of Big Data?
  Grounding    : 0.667
  Citations OK : 1/1
  Found        : ['volume, velocity, variety']
  Missing      : ['datasets too large for traditional tools', 'scalable processing']

Question 3: What challenges does Big Data pose in terms of data processing and analysis?
  Grounding    : 0.923
  Citations OK : 1/1
  Found        : []
  Missing      : ['volume, velocity, variety', 'datasets too large for traditional tools', 'scalable processing']

Expected elements:
  - volume, velocity, variety
  - datasets too large for traditional tools
  - scalable processing


## Format-adherence metrics

In [64]:
# ================================================================
# SECTION 5 — Format-Adherence Metrics
# ================================================================

def evaluate_format_adherence(result: dict) -> dict:
    """
    Évalue si l'output LLM respecte le format JSON attendu.
    Métriques :
    - parse_success      : JSON parseable sans modification ?
    - required_fields    : tous les champs requis présents ?
    - citation_format    : citations au bon format ?
    - refusal_valid      : si refus, est-il bien formaté ?
    - overall_valid      : tout est valide ?
    """
    raw_output    = result["raw_output"]
    parsed_output = result["parsed_output"]
    parse_success = result["parse_success"]
    questions     = parsed_output.get("questions", [])
    refusal       = parsed_output.get("refusal")

    # ── 1. Parse success ──────────────────────────────────────
    # Déjà calculé dans le pipeline

    # ── 2. Required fields au niveau racine ───────────────────
    required_root_fields = {"query", "questions", "refusal"}
    present_root = set(parsed_output.keys())
    missing_root = required_root_fields - present_root
    root_fields_ok = len(missing_root) == 0

    # ── 3. Required fields par question ───────────────────────
    required_q_fields = {"question", "answer", "citations"}
    question_field_results = []

    for q in questions:
        present = set(q.keys())
        missing = required_q_fields - present
        question_field_results.append({
            "ok":     len(missing) == 0,
            "missing": list(missing)
        })

    questions_fields_ok = all(r["ok"] for r in question_field_results)
    questions_field_rate = (
        sum(1 for r in question_field_results if r["ok"]) / len(question_field_results)
        if question_field_results else 1.0
    )

    # ── 4. Citation format validity ───────────────────────────
    # Format attendu : contient pdf_name ou p.X ou chunk_id
    import re
    citation_pattern = re.compile(
        r"(\.pdf|p\.\d+|chunk_id|chunk|\[\d+\])", re.IGNORECASE
    )

    all_citations = []
    for q in questions:
        all_citations.extend(q.get("citations", []))

    valid_citation_formats = [
        bool(citation_pattern.search(str(c))) for c in all_citations
    ]
    citation_format_rate = (
        sum(valid_citation_formats) / len(valid_citation_formats)
        if valid_citation_formats else 1.0
    )

    # ── 5. Refusal format validity ────────────────────────────
    # Si refus : questions doit être vide et refusal doit être une string
    refusal_valid = True
    if refusal is not None:
        refusal_valid = (
            isinstance(refusal, str) and
            len(refusal) > 0 and
            len(questions) == 0
        )

    # ── 6. Overall validity ───────────────────────────────────
    overall_valid = (
        parse_success and
        root_fields_ok and
        questions_fields_ok and
        citation_format_rate >= 0.5
    )

    return {
        "query":                result["query"],
        "parse_success":        parse_success,
        "root_fields_ok":       root_fields_ok,
        "missing_root_fields":  list(missing_root),
        "questions_field_rate": round(questions_field_rate, 3),
        "citation_format_rate": round(citation_format_rate, 3),
        "n_citations":          len(all_citations),
        "refusal_valid":        refusal_valid,
        "overall_valid":        overall_valid,
    }


# Évaluer tous les résultats
format_evals = [evaluate_format_adherence(r) for r in results]

# ── Résumé ────────────────────────────────────────────────────
n_total   = len(format_evals)
n_parsed  = sum(1 for e in format_evals if e["parse_success"])
n_valid   = sum(1 for e in format_evals if e["overall_valid"])
n_root_ok = sum(1 for e in format_evals if e["root_fields_ok"])

print("=" * 60)
print("FORMAT-ADHERENCE METRICS")
print("=" * 60)
print(f"Parser pass rate        : {n_parsed}/{n_total} = {n_parsed/n_total:.1%}")
print(f"Overall valid rate      : {n_valid}/{n_total}  = {n_valid/n_total:.1%}")
print(f"Root fields OK          : {n_root_ok}/{n_total} = {n_root_ok/n_total:.1%}")
print(f"Avg question field rate : {sum(e['questions_field_rate'] for e in format_evals)/n_total:.1%}")
print(f"Avg citation format rate: {sum(e['citation_format_rate'] for e in format_evals)/n_total:.1%}")

# ── Détail par query ──────────────────────────────────────────
print(f"\n{'Query':<45} | {'Parse':>5} | {'Fields':>6} | {'Cite':>5} | {'Valid':>5}")
print("-" * 75)
for e in format_evals:
    print(f"{e['query'][:45]:<45} | "
          f"{'✅' if e['parse_success'] else '❌':>5} | "
          f"{e['questions_field_rate']:>6.1%} | "
          f"{e['citation_format_rate']:>5.1%} | "
          f"{'✅' if e['overall_valid'] else '❌':>5}")

FORMAT-ADHERENCE METRICS
Parser pass rate        : 1/1 = 100.0%
Overall valid rate      : 1/1  = 100.0%
Root fields OK          : 1/1 = 100.0%
Avg question field rate : 100.0%
Avg citation format rate: 100.0%

Query                                         | Parse | Fields |  Cite | Valid
---------------------------------------------------------------------------
What is Big Data and what are its main charac |     ✅ | 100.0% | 100.0% |     ✅


## System metrics

In [65]:
# ================================================================
# SECTION 6 — System Metrics
# ================================================================

def compute_system_metrics(results: list) -> dict:
    """
    Agrège toutes les métriques système collectées pendant le pipeline.
    """
    n = len(results)
    if n == 0:
        return {}

    latencies_total    = [r["total_time_s"]       for r in results]
    latencies_retrieve = [r["retrieval_time_s"]    for r in results]
    latencies_generate = [r["generation_time_s"]   for r in results]
    tokens_in          = [r["tokens_in"]           for r in results]
    tokens_out         = [r["tokens_out"]          for r in results]
    memory_deltas      = [r["memory_delta_mb"]     for r in results]

    return {
        # Latence (secondes)
        "latency": {
            "mean_total_s":      round(sum(latencies_total)    / n, 3),
            "mean_retrieval_s":  round(sum(latencies_retrieve) / n, 3),
            "mean_generation_s": round(sum(latencies_generate) / n, 3),
            "min_total_s":       round(min(latencies_total), 3),
            "max_total_s":       round(max(latencies_total), 3),
        },
        # Tokens
        "tokens": {
            "mean_tokens_in":    round(sum(tokens_in)  / n, 1),
            "mean_tokens_out":   round(sum(tokens_out) / n, 1),
            "total_tokens_in":   sum(tokens_in),
            "total_tokens_out":  sum(tokens_out),
        },
        # Throughput
        "throughput": {
            "queries_per_minute": round(60 / (sum(latencies_total) / n), 2),
            "tokens_per_second":  round(
                sum(tokens_out) / sum(latencies_generate), 2
            ),
        },
        # Mémoire
        "memory": {
            "mean_delta_mb": round(sum(memory_deltas) / n, 1),
            "max_delta_mb":  round(max(memory_deltas), 1),
        },
        # Coût
        "cost": {
            "model":        "llama3.2:3b (local)",
            "monetary_usd": 0.0,
            "note":         "Local model — no API cost. Token and runtime costs reported above."
        },
        "n_queries": n,
    }


# Calculer les métriques
sys_metrics = compute_system_metrics(results)

# ── Affichage ─────────────────────────────────────────────────
print("=" * 55)
print("SYSTEM METRICS")
print("=" * 55)

print("\n── Latency ──────────────────────────────────────────")
print(f"  Mean total latency    : {sys_metrics['latency']['mean_total_s']}s")
print(f"  Mean retrieval time   : {sys_metrics['latency']['mean_retrieval_s']}s")
print(f"  Mean generation time  : {sys_metrics['latency']['mean_generation_s']}s")
print(f"  Min / Max total       : {sys_metrics['latency']['min_total_s']}s / {sys_metrics['latency']['max_total_s']}s")

print("\n── Tokens ───────────────────────────────────────────")
print(f"  Mean tokens in        : {sys_metrics['tokens']['mean_tokens_in']}")
print(f"  Mean tokens out       : {sys_metrics['tokens']['mean_tokens_out']}")
print(f"  Total tokens in       : {sys_metrics['tokens']['total_tokens_in']}")
print(f"  Total tokens out      : {sys_metrics['tokens']['total_tokens_out']}")

print("\n── Throughput ───────────────────────────────────────")
print(f"  Queries / minute      : {sys_metrics['throughput']['queries_per_minute']}")
print(f"  Tokens / second       : {sys_metrics['throughput']['tokens_per_second']}")

print("\n── Memory ───────────────────────────────────────────")
print(f"  Mean memory delta     : {sys_metrics['memory']['mean_delta_mb']} MB")
print(f"  Max memory delta      : {sys_metrics['memory']['max_delta_mb']} MB")

print("\n── Cost ─────────────────────────────────────────────")
print(f"  Model                 : {sys_metrics['cost']['model']}")
print(f"  Monetary cost         : ${sys_metrics['cost']['monetary_usd']}")
print(f"  Note                  : {sys_metrics['cost']['note']}")

print("\n── Per-query breakdown ──────────────────────────────")
print(f"{'Query':<45} | {'Total':>7} | {'Retr':>6} | {'Gen':>6} | {'Tok_in':>6} | {'Tok_out':>7}")
print("-" * 90)
for r in results:
    print(f"{r['query'][:45]:<45} | "
          f"{r['total_time_s']:>7.2f}s | "
          f"{r['retrieval_time_s']:>6.3f}s | "
          f"{r['generation_time_s']:>6.2f}s | "
          f"{r['tokens_in']:>6} | "
          f"{r['tokens_out']:>7}")

SYSTEM METRICS

── Latency ──────────────────────────────────────────
  Mean total latency    : 23.81s
  Mean retrieval time   : 0.014s
  Mean generation time  : 23.796s
  Min / Max total       : 23.81s / 23.81s

── Tokens ───────────────────────────────────────────
  Mean tokens in        : 380.0
  Mean tokens out       : 99.0
  Total tokens in       : 380
  Total tokens out      : 99

── Throughput ───────────────────────────────────────
  Queries / minute      : 2.52
  Tokens / second       : 4.16

── Memory ───────────────────────────────────────────
  Mean memory delta     : 0.0 MB
  Max memory delta      : 0.0 MB

── Cost ─────────────────────────────────────────────
  Model                 : llama3.2:3b (local)
  Monetary cost         : $0.0
  Note                  : Local model — no API cost. Token and runtime costs reported above.

── Per-query breakdown ──────────────────────────────
Query                                         |   Total |   Retr |    Gen | Tok_in | Tok_out


## Regression testing

In [70]:
# ================================================================
# SECTION 7 — Regression Testing
# ================================================================

import json
from pathlib import Path
from datetime import datetime

# Prompts fixes qui ne changent jamais
REGRESSION_PROMPTS = [
    "What is MapReduce and how does it work?",
    "How does HDFS handle fault tolerance?",
    "What is the CAP theorem?",
    "Explain RDD lineage in Spark.",
    "How does LSH work for similarity search?",
]

def run_regression_test(
    prompts: list,
    chunks: list,
    doc_matrix,
    embedding_strategy: str,
    similarity_metric: str,
    k: int = 5,
    sbert_model=None,
    tfidf_vectorizer=None,
    w2v_model=None,
    model: str = "llama3.2:3b",
    save_path: str = "/home/jovyan/work/regression_baseline.json"
) -> dict:
    """
    Lance le pipeline sur les prompts fixes et sauvegarde les résultats.
    Si une baseline existe déjà, compare avec elle.
    """
    print(f"Running {len(prompts)} regression tests...")
    
    current_run = {
        "timestamp":          datetime.now().isoformat(),
        "embedding_strategy": embedding_strategy,
        "similarity_metric":  similarity_metric,
        "k":                  k,
        "model":              model,
        "results":            {}
    }
    
    for prompt in prompts:
        print(f"  → {prompt[:60]}...")
        result = run_generation_pipeline(
            query=prompt,
            chunks=chunks,
            relevant_pages=[3, 7, 14],
            doc_matrix=doc_matrix,
            embedding_strategy=embedding_strategy,
            similarity_metric=similarity_metric,
            k=k,
            n_questions=2,   # 2 questions suffit pour les tests de régression
            model=model,
            sbert_model=sbert_model,
            tfidf_vectorizer=tfidf_vectorizer,
            w2v_model=w2v_model,
        )
        
        # Garder uniquement ce qui est nécessaire pour la comparaison
        current_run["results"][prompt] = {
            "parse_success":  result["parse_success"],
            "n_questions":    result["n_questions"],
            "tokens_in":      result["tokens_in"],
            "tokens_out":     result["tokens_out"],
            "total_time_s":   result["total_time_s"],
            "questions":      result["parsed_output"].get("questions", []),
            "refusal":        result["parsed_output"].get("refusal"),
        }
    
    return current_run


def compare_with_baseline(current_run: dict, save_path: str) -> dict:
    """
    Compare le run actuel avec la baseline sauvegardée.
    Sauvegarde le run actuel comme nouvelle baseline si aucune n'existe.
    """
    path = Path(save_path)
    
    # ── Pas de baseline → sauvegarder et retourner ────────────
    if not path.exists():
        with open(path, "w") as f:
            json.dump(current_run, f, indent=2)
        print(f"✅ Baseline saved to {save_path}")
        return {"status": "baseline_created", "comparisons": []}
    
    # ── Baseline existe → comparer ────────────────────────────
    with open(path) as f:
        baseline = json.load(f)
    
    print(f"\nBaseline from : {baseline['timestamp']}")
    print(f"Current run   : {current_run['timestamp']}")
    
    comparisons = []
    regressions = []
    improvements = []
    
    for prompt in current_run["results"]:
        if prompt not in baseline["results"]:
            print(f"  ⚠️  New prompt (not in baseline): {prompt[:50]}")
            continue
        
        curr = current_run["results"][prompt]
        base = baseline["results"][prompt]
        
        # Comparer les métriques clés
        parse_ok      = curr["parse_success"] == base["parse_success"]
        n_q_ok        = curr["n_questions"]   >= base["n_questions"] - 1
        time_delta    = curr["total_time_s"]   - base["total_time_s"]
        tokens_delta  = curr["tokens_out"]     - base["tokens_out"]

        status = "✅ OK"
        if not parse_ok:
            status = "❌ REGRESSION: parse failed"
            regressions.append(prompt)
        elif not n_q_ok:
            status = "⚠️  REGRESSION: fewer questions"
            regressions.append(prompt)
        elif curr["n_questions"] > base["n_questions"]:
            status = "⬆️  IMPROVEMENT: more questions"
            improvements.append(prompt)

        comparisons.append({
            "prompt":       prompt,
            "status":       status,
            "parse":        f"{'✅' if curr['parse_success'] else '❌'} "
                           f"(was {'✅' if base['parse_success'] else '❌'})",
            "n_questions":  f"{curr['n_questions']} (was {base['n_questions']})",
            "time_delta_s": round(time_delta, 2),
            "tokens_delta": tokens_delta,
        })
    
    return {
        "status":       "compared",
        "n_regressions":   len(regressions),
        "n_improvements":  len(improvements),
        "n_stable":        len(comparisons) - len(regressions) - len(improvements),
        "regressions":     regressions,
        "improvements":    improvements,
        "comparisons":     comparisons,
    }


def display_regression_report(comparison: dict) -> None:
    """Affiche un rapport lisible de la comparaison."""
    
    if comparison["status"] == "baseline_created":
        print("Baseline created — no comparison available yet.")
        print("Re-run after any change to see regressions.")
        return
    
    print("\n" + "=" * 60)
    print("REGRESSION TEST REPORT")
    print("=" * 60)
    print(f"  Regressions  : {comparison['n_regressions']}")
    print(f"  Improvements : {comparison['n_improvements']}")
    print(f"  Stable       : {comparison['n_stable']}")
    
    print(f"\n{'Prompt':<45} | {'Status':<35} | {'Questions':>9} | {'Δ Time':>7}")
    print("-" * 105)
    for c in comparison["comparisons"]:
        print(f"{c['prompt'][:45]:<45} | "
              f"{c['status']:<35} | "
              f"{c['n_questions']:>9} | "
              f"{c['time_delta_s']:>+7.1f}s")
    
    if comparison["n_regressions"] > 0:
        print(f"\n⚠️  {comparison['n_regressions']} regression(s) detected!")
        for r in comparison["regressions"]:
            print(f"  → {r}")
    else:
        print("\n✅ No regressions detected.")


# ── Lancer les tests ──────────────────────────────────────────
REGRESSION_SAVE_PATH = "/home/jovyan/work/regression_baseline.json"

current_run = run_regression_test(
    prompts=REGRESSION_PROMPTS,
    chunks=chunks,
    doc_matrix=sbert_emb,
    embedding_strategy="sbert",
    similarity_metric="cosine",
    k=3,
    sbert_model=sbert_model,
    model="llama3.2:3b",
    save_path=REGRESSION_SAVE_PATH,
)

comparison = compare_with_baseline(current_run, REGRESSION_SAVE_PATH)
display_regression_report(comparison)

Running 5 regression tests...
  → What is MapReduce and how does it work?...
  → How does HDFS handle fault tolerance?...
  → What is the CAP theorem?...
  → Explain RDD lineage in Spark....
  → How does LSH work for similarity search?...

Baseline from : 2026-05-21T10:27:19.779869
Current run   : 2026-05-21T10:33:08.156612

REGRESSION TEST REPORT
  Regressions  : 0
  Improvements : 1
  Stable       : 4

Prompt                                        | Status                              | Questions |  Δ Time
---------------------------------------------------------------------------------------------------------
What is MapReduce and how does it work?       | ✅ OK                                | 2 (was 2) |   -15.5s
How does HDFS handle fault tolerance?         | ✅ OK                                | 2 (was 2) |    -8.9s
What is the CAP theorem?                      | ⬆️  IMPROVEMENT: more questions     | 3 (was 2) |    -7.9s
Explain RDD lineage in Spark.                 | ✅ OK       

## Options 2 et 3 

## Option 1 — throughput avec différents numSlices

## Option 4 — HNSW avec faiss

## Manual evaluation

In [71]:
# ================================================================
# SECTION 8 — Manual Quality Evaluation
# ================================================================

# Critères de notation (0 = mauvais, 1 = moyen, 2 = bon)
RUBRIC = {
    "relevance":        "Question liée au topic et aux chunks récupérés ?",
    "correctness":      "Techniquement correcte par rapport au cours ?",
    "faithfulness":     "Supportée par les chunks, sans hallucination ?",
    "citation_support": "Citations pointent vers les bons chunks ?",
    "clarity":          "Question bien formulée, appropriée pour un examen ?",
    "format_adherence": "Output suit le format requis (JSON, champs présents) ?",
}

def display_for_manual_evaluation(results: list, n_samples: int = 5) -> None:
    """
    Affiche les questions générées pour évaluation manuelle.
    Sélectionne un sous-ensemble représentatif.
    """
    import random
    # Prendre les n_samples premiers résultats (ou tous si moins)
    samples = results[:n_samples]
    
    for r_idx, result in enumerate(samples):
        print(f"\n{'='*70}")
        print(f"[{r_idx+1}/{len(samples)}] QUERY: {result['query']}")
        print(f"Embedding: {result['embedding_strategy']} | "
              f"Metric: {result['similarity_metric']} | k={result['k']}")
        print(f"\nRetrieved chunks:")
        for i, chunk in enumerate(result["retrieved_chunks"][:3]):
            c = chunk["chunk"]
            print(f"  [{i+1}] {c['pdf_name']} p.{c['page_num']} | "
                  f"score={chunk['score']:.3f}")
            print(f"       {c['text'][:100]}...")
        
        print(f"\nGenerated questions:")
        questions = result["parsed_output"].get("questions", [])
        for q_idx, q in enumerate(questions):
            print(f"\n  Q{q_idx+1}: {q['question']}")
            print(f"  A : {q['answer'][:200]}...")
            print(f"  Citations: {q['citations']}")
        
        print(f"\n{'─'*70}")
        print("RUBRIC SCORES (fill manually — 0=bad, 1=ok, 2=good):")
        print(f"  {'Criterion':<20} | {'Score':>5} | Description")
        print(f"  {'-'*65}")
        for criterion, description in RUBRIC.items():
            print(f"  {criterion:<20} | {'?/2':>5} | {description}")


# Afficher pour évaluation manuelle
display_for_manual_evaluation(results, n_samples=min(5, len(results)))


[1/1] QUERY: What is Big Data and what are its main characteristics?
Embedding: sbert | Metric: cosine | k=4

Retrieved chunks:
  [1] 1-intro.pdf p.3 | score=0.917
       WHAT IS BIG DATA?...
  [2] 1-intro.pdf p.18 | score=0.756
       BIG DATA: SOME DEFINITIONS (4/4) Big data refers to extremely large datasets that cannot be easily m...
  [3] 1-intro.pdf p.17 | score=0.755
       BIG DATA: SOME DEFINITIONS (3/4) Big data are data sets that are so voluminous and complex that trad...

Generated questions:

  Q1: What does the term 'Big Data' refer to in terms of its size?
  A : extremely large datasets...
  Citations: ['1-intro.pdf p.3 chunk_id:p3_c0']

  Q2: What are the three main characteristics of Big Data?
  A : volume, variety, and velocity...
  Citations: ['2-intro.pdf p.18 chunk_id:p18_c0']

  Q3: What challenges does Big Data pose in terms of data processing and analysis?
  A : capturing data, data storage, data analysis, search, sharing, transfer, visualization, querying, upd

In [ ]:
chunks["scores"]intro.pdf p.18 chunk_id:p18_c0

In [72]:
# Après avoir noté manuellement, remplissez ce tableau :
MANUAL_SCORES = [
    {
        "query":    "What is Big Data and what are its main characteristics?",
        "question": "What are the three main characteristics of Big Data?",
        "scores": {
            "relevance":        2,   # ← remplir 0, 1, ou 2
            "correctness":      2,
            "faithfulness":     2,
            "citation_support": 1,
            "clarity":          2,
            "format_adherence": 2,
        },
        "notes": "Good question, answer slightly incomplete on scalability aspect."
    },
    # Ajouter les autres questions ici...
]

def compute_manual_scores_summary(manual_scores: list) -> None:
    """Affiche le résumé des scores manuels."""
    
    if not manual_scores:
        print("No manual scores yet.")
        return
    
    criteria = list(RUBRIC.keys())
    
    print("\n" + "=" * 70)
    print("MANUAL QUALITY EVALUATION SUMMARY")
    print("=" * 70)
    
    # Score par critère
    print(f"\n{'Criterion':<20} | {'Mean':>6} | {'Min':>4} | {'Max':>4} | {'%Good (=2)':>10}")
    print("-" * 60)
    
    for criterion in criteria:
        scores = [s["scores"][criterion] for s in manual_scores 
                  if criterion in s["scores"]]
        if not scores:
            continue
        mean  = sum(scores) / len(scores)
        pct_good = sum(1 for s in scores if s == 2) / len(scores)
        print(f"  {criterion:<18} | {mean:>6.2f} | {min(scores):>4} | "
              f"{max(scores):>4} | {pct_good:>10.1%}")
    
    # Score global par question
    print(f"\n{'Question':<50} | {'Total':>5} | {'%':>5}")
    print("-" * 65)
    for item in manual_scores:
        total = sum(item["scores"].values())
        max_score = len(item["scores"]) * 2
        pct = total / max_score
        print(f"  {item['question'][:48]:<48} | {total:>4}/{max_score} | {pct:>5.1%}")
        if item.get("notes"):
            print(f"    Note: {item['notes']}")
    
    # Moyenne globale
    all_totals = [sum(s["scores"].values()) for s in manual_scores]
    max_total  = len(criteria) * 2
    global_mean = sum(all_totals) / len(all_totals)
    print(f"\n  Overall mean score : {global_mean:.2f}/{max_total} "
          f"({global_mean/max_total:.1%})")


compute_manual_scores_summary(MANUAL_SCORES)


MANUAL QUALITY EVALUATION SUMMARY

Criterion            |   Mean |  Min |  Max | %Good (=2)
------------------------------------------------------------
  relevance          |   2.00 |    2 |    2 |     100.0%
  correctness        |   2.00 |    2 |    2 |     100.0%
  faithfulness       |   2.00 |    2 |    2 |     100.0%
  citation_support   |   1.00 |    1 |    1 |       0.0%
  clarity            |   2.00 |    2 |    2 |     100.0%
  format_adherence   |   2.00 |    2 |    2 |     100.0%

Question                                           | Total |     %
-----------------------------------------------------------------
  What are the three main characteristics of Big D |   11/12 | 91.7%
    Note: Good question, answer slightly incomplete on scalability aspect.

  Overall mean score : 11.00/12 (91.7%)


## Regression testing